In [1]:
# step1_verify_dataset.py
import os
from PIL import Image

DATASET_DIR = "EARTH-STUDY-DATASET"   # apna path adjust kar lena agar different hai

def verify_dataset(root_dir):
    if not os.path.exists(root_dir):
        print(f"ERROR: '{root_dir}' path nahi mila. Path check karo.")
        return

    class_folders = sorted([
        d for d in os.listdir(root_dir)
        if os.path.isdir(os.path.join(root_dir, d))
    ])

    print(f"Total classes found: {len(class_folders)}\n")
    print(f"{'Class':<20}{'Images':<10}{'Corrupt':<10}{'Sizes (unique)'}")
    print("-" * 60)

    total_images = 0
    total_corrupt = 0

    for cls in class_folders:
        cls_path = os.path.join(root_dir, cls)
        files = [f for f in os.listdir(cls_path)
                 if f.lower().endswith(('.jpg', '.jpeg', '.png', '.tif', '.tiff'))]

        corrupt = 0
        sizes = set()
        for f in files:
            fpath = os.path.join(cls_path, f)
            try:
                with Image.open(fpath) as img:
                    img.verify()
                with Image.open(fpath) as img:
                    sizes.add(img.size)
            except Exception:
                corrupt += 1

        total_images += len(files)
        total_corrupt += corrupt

        print(f"{cls:<20}{len(files):<10}{corrupt:<10}{sizes if len(sizes) <= 3 else str(list(sizes)[:3]) + '...'}")

    print("-" * 60)
    print(f"TOTAL images: {total_images}")
    print(f"TOTAL corrupt: {total_corrupt}")
    print(f"TOTAL classes: {len(class_folders)}")

if __name__ == "__main__":
    verify_dataset(DATASET_DIR)

Total classes found: 21

Class               Images    Corrupt   Sizes (unique)
------------------------------------------------------------
agricultural        100       0         {(256, 256)}
baseballdiamond     100       0         [(252, 256), (256, 255), (250, 256)]...
buildings           100       0         {(256, 256), (247, 247)}
denseresidential    100       0         {(256, 256)}
harbor              100       0         {(257, 257), (253, 256), (256, 256)}
mediumresidential   100       0         {(256, 256)}
mobilehomepark      100       0         {(253, 256), (256, 253), (256, 256)}
parkinglot          100       0         {(255, 256), (256, 256), (256, 247)}
runway              100       0         {(256, 254), (256, 256), (254, 256)}
sparseresidential   100       0         [(250, 256), (255, 256), (256, 256)]...
storagetanks        100       0         {(256, 249), (256, 256), (254, 256)}
tenniscourt         100       0         {(255, 256), (247, 256), (256, 256)}
uc_airplane  

In [2]:
# step2_prepare_data.py
import os
import shutil
import random
from PIL import Image

DATASET_DIR = "EARTH-STUDY-DATASET"
OUTPUT_DIR = "split_dataset"

SPLIT_RATIO = {"train": 0.70, "val": 0.15, "test": 0.15}
SEED = 42
IMG_SIZE = (224, 224)   # paper ke hisaab se 224x224

random.seed(SEED)

def prepare_splits():
    if os.path.exists(OUTPUT_DIR):
        shutil.rmtree(OUTPUT_DIR)

    classes = sorted([
        d for d in os.listdir(DATASET_DIR)
        if os.path.isdir(os.path.join(DATASET_DIR, d))
    ])

    summary = {}

    for cls in classes:
        cls_path = os.path.join(DATASET_DIR, cls)
        files = [f for f in os.listdir(cls_path)
                 if f.lower().endswith(('.jpg', '.jpeg', '.png', '.tif', '.tiff'))]
        random.shuffle(files)

        n = len(files)
        n_train = int(n * SPLIT_RATIO["train"])
        n_val = int(n * SPLIT_RATIO["val"])
        # remaining goes to test
        splits = {
            "train": files[:n_train],
            "val": files[n_train:n_train + n_val],
            "test": files[n_train + n_val:]
        }

        for split_name, split_files in splits.items():
            out_dir = os.path.join(OUTPUT_DIR, split_name, cls)
            os.makedirs(out_dir, exist_ok=True)
            for f in split_files:
                src = os.path.join(cls_path, f)
                dst = os.path.join(out_dir, f)
                # resize + save as consistent RGB, fixes minor dimension mismatches
                try:
                    with Image.open(src) as img:
                        img = img.convert("RGB").resize(IMG_SIZE)
                        img.save(dst)
                except Exception as e:
                    print(f"Skipping {src}: {e}")

        summary[cls] = {k: len(v) for k, v in splits.items()}

    print(f"{'Class':<20}{'Train':<8}{'Val':<8}{'Test':<8}")
    print("-" * 44)
    total = {"train": 0, "val": 0, "test": 0}
    for cls, counts in summary.items():
        print(f"{cls:<20}{counts['train']:<8}{counts['val']:<8}{counts['test']:<8}")
        for k in total:
            total[k] += counts[k]
    print("-" * 44)
    print(f"{'TOTAL':<20}{total['train']:<8}{total['val']:<8}{total['test']:<8}")
    print(f"\nSplit dataset saved to: {OUTPUT_DIR}/")

if __name__ == "__main__":
    prepare_splits()

Class               Train   Val     Test    
--------------------------------------------
agricultural        70      15      15      
baseballdiamond     70      15      15      
buildings           70      15      15      
denseresidential    70      15      15      
harbor              70      15      15      
mediumresidential   70      15      15      
mobilehomepark      70      15      15      
parkinglot          70      15      15      
runway              70      15      15      
sparseresidential   70      15      15      
storagetanks        70      15      15      
tenniscourt         70      15      15      
uc_airplane         70      15      15      
uc_beach            70      15      15      
uc_chaparral        70      15      15      
uc_forest           70      15      15      
uc_freeway          70      15      15      
uc_golfcourse       70      15      15      
uc_intersection     70      15      15      
uc_overpass         70      15      15      
uc_river  

In [3]:
# step3_baseline_inceptionv3.py
import tensorflow as tf
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.layers import Dense, Dropout, Flatten, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator

DATA_DIR = "split_dataset"
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 50
NUM_CLASSES = 21

# ---- Data generators (paper ke augmentation: flip, 15 rotation, 0.2 zoom) ----
train_datagen = ImageDataGenerator(
    rescale=1./255,
    horizontal_flip=True,
    rotation_range=15,
    zoom_range=0.2
)
val_test_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    f"{DATA_DIR}/train", target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="categorical", shuffle=True
)
val_gen = val_test_datagen.flow_from_directory(
    f"{DATA_DIR}/val", target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="categorical", shuffle=False
)
test_gen = val_test_datagen.flow_from_directory(
    f"{DATA_DIR}/test", target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="categorical", shuffle=False
)

print("Class indices:", train_gen.class_indices)

# ---- Base model: Inception-V3, ImageNet pretrained, top layers frozen ----
base_model = InceptionV3(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False   # freeze all pretrained layers (paper's approach)

# ---- Paper's proposed FC head (Figure 3) ----
x = base_model.output
x = GlobalAveragePooling2D()(x)   # Flatten alternative, more stable for Inception
x = Dense(1024, activation="relu")(x)
x = Dropout(0.5)(x)
x = Dense(512, activation="relu")(x)
x = Dropout(0.5)(x)
output = Dense(NUM_CLASSES, activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

# ---- Training ----
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    "baseline_inceptionv3_best.h5", monitor="val_accuracy",
    save_best_only=True, mode="max"
)
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_accuracy", patience=10, restore_best_weights=True
)

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=[checkpoint, early_stop]
)

# ---- Evaluate on test set ----
test_loss, test_acc = model.evaluate(test_gen)
print(f"\nBASELINE Inception-V3 Test Accuracy: {test_acc*100:.2f}%")

model.save("baseline_inceptionv3_final.h5")

Found 1470 images belonging to 21 classes.
Found 315 images belonging to 21 classes.
Found 315 images belonging to 21 classes.
Class indices: {'agricultural': 0, 'baseballdiamond': 1, 'buildings': 2, 'denseresidential': 3, 'harbor': 4, 'mediumresidential': 5, 'mobilehomepark': 6, 'parkinglot': 7, 'runway': 8, 'sparseresidential': 9, 'storagetanks': 10, 'tenniscourt': 11, 'uc_airplane': 12, 'uc_beach': 13, 'uc_chaparral': 14, 'uc_forest': 15, 'uc_freeway': 16, 'uc_golfcourse': 17, 'uc_intersection': 18, 'uc_overpass': 19, 'uc_river': 20}
87910968/87910968 ━━━━━━━━━━━━━━━━━━━━ 96s 1us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 111, 111,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 111, 111,  │         96 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 111, 111,  │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 109, 109,  │      9,216 │ activation[0][0]  │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 109, 109,  │         96 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 109, 109,  │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 109, 109,  │     18,432 │ activation_1[0][… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 109, 109,  │        192 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 109, 109,  │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 54, 54,    │          0 │ activation_2[0][… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 54, 54,    │      5,120 │ max_pooling2d[0]… │
│                     │ 80)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 54, 54,    │        240 │ conv2d_3[0][0]    │
│ (BatchNormalizatio… │ 80)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_3        │ (None, 54, 54,    │          0 │ batch_normalizat… │
│ (Activation)        │ 80)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 52, 52,    │    138,240 │ activation_3[0][… │
│                     │ 192)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 52, 52,    │        576 │ conv2d_4[0][0]    │
│ (BatchNormalizatio… │ 192)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_4        │ (None, 52, 52,    │          0 │ batch_normalizat

 Total params: 24,436,533 (93.22 MB)

 Trainable params: 2,633,749 (10.05 MB)

 Non-trainable params: 21,802,784 (83.17 MB)

... (9 outputs left out)

46/46 ━━━━━━━━━━━━━━━━━━━━ 47s 1s/step - accuracy: 0.8755 - loss: 0.3600 - val_accuracy: 0.9270 - val_loss: 0.2474
Epoch 21/50
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 845ms/step - accuracy: 0.9020 - loss: 0.3028

46/46 ━━━━━━━━━━━━━━━━━━━━ 47s 1s/step - accuracy: 0.9020 - loss: 0.3028 - val_accuracy: 0.9302 - val_loss: 0.2559
Epoch 22/50
46/46 ━━━━━━━━━━━━━━━━━━━━ 47s 1s/step - accuracy: 0.9007 - loss: 0.3043 - val_accuracy: 0.9270 - val_loss: 0.2378
Epoch 23/50
46/46 ━━━━━━━━━━━━━━━━━━━━ 46s 1s/step - accuracy: 0.9054 - loss: 0.3125 - val_accuracy: 0.9143 - val_loss: 0.2299
Epoch 24/50
46/46 ━━━━━━━━━━━━━━━━━━━━ 46s 1s/step - accuracy: 0.8980 - loss: 0.3039 - val_accuracy: 0.9270 - val_loss: 0.2285
Epoch 25/50
46/46 ━━━━━━━━━━━━━━━━━━━━ 47s 1s/step - accuracy: 0.9048 - loss: 0.2911 - val_accuracy: 0.9302 - val_loss: 0.2205
Epoch 26/50
46/46 ━━━━━━━━━━━━━━━━━━━━ 47s 1s/step - accuracy: 0.8986 - loss: 0.3073 - val_accuracy: 0.9175 - val_loss: 0.2290
Epoch 27/50
46/46 ━━━━━━━━━━━━━━━━━━━━ 47s 1s/step - accuracy: 0.9265 - loss: 0.2526 - val_accuracy: 0.9238 - val_loss: 0.2335
Epoch 28/50
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 855ms/step - accuracy: 0.9245 - loss: 0.2688

46/46 ━━━━━━━━━━━━━━━━━━━━ 48s 1s/step - accuracy: 0.9245 - loss: 0.2688 - val_accuracy: 0.9397 - val_loss: 0.2184
Epoch 29/50
46/46 ━━━━━━━━━━━━━━━━━━━━ 47s 1s/step - accuracy: 0.9252 - loss: 0.2555 - val_accuracy: 0.9302 - val_loss: 0.2135
Epoch 30/50
46/46 ━━━━━━━━━━━━━━━━━━━━ 47s 1s/step - accuracy: 0.9143 - loss: 0.2494 - val_accuracy: 0.9238 - val_loss: 0.2094
Epoch 31/50
46/46 ━━━━━━━━━━━━━━━━━━━━ 47s 1s/step - accuracy: 0.9184 - loss: 0.2548 - val_accuracy: 0.9270 - val_loss: 0.2499
Epoch 32/50
46/46 ━━━━━━━━━━━━━━━━━━━━ 47s 1s/step - accuracy: 0.9259 - loss: 0.2325 - val_accuracy: 0.9270 - val_loss: 0.2124
Epoch 33/50
46/46 ━━━━━━━━━━━━━━━━━━━━ 55s 1s/step - accuracy: 0.9286 - loss: 0.2092 - val_accuracy: 0.9302 - val_loss: 0.2202
Epoch 34/50
46/46 ━━━━━━━━━━━━━━━━━━━━ 54s 1s/step - accuracy: 0.9361 - loss: 0.1906 - val_accuracy: 0.9365 - val_loss: 0.2022
Epoch 35/50
46/46 ━━━━━━━━━━━━━━━━━━━━ 51s 1s/step - accuracy: 0.9347 - loss: 0.2127 - val_accuracy: 0.9206 - val_loss: 0.2


BASELINE Inception-V3 Test Accuracy: 92.38%


In [5]:
# step4_improved_cbam_inceptionv3.py
import tensorflow as tf
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.layers import (
    Dense, Dropout, GlobalAveragePooling2D, GlobalMaxPooling2D,
    Reshape, Multiply, Concatenate, Conv2D, Add, Activation, Lambda
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator

DATA_DIR = "split_dataset"
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 50
NUM_CLASSES = 21

train_datagen = ImageDataGenerator(
    rescale=1./255,
    horizontal_flip=True,
    rotation_range=15,
    zoom_range=0.2
)
val_test_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    f"{DATA_DIR}/train", target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="categorical", shuffle=True
)
val_gen = val_test_datagen.flow_from_directory(
    f"{DATA_DIR}/val", target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="categorical", shuffle=False
)
test_gen = val_test_datagen.flow_from_directory(
    f"{DATA_DIR}/test", target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="categorical", shuffle=False
)

def channel_attention(input_feature, ratio=8):
    channel = input_feature.shape[-1]

    shared_dense_one = Dense(channel // ratio, activation="relu",
                              kernel_initializer="he_normal", use_bias=True)
    shared_dense_two = Dense(channel, kernel_initializer="he_normal", use_bias=True)

    avg_pool = GlobalAveragePooling2D()(input_feature)
    avg_pool = Reshape((1, 1, channel))(avg_pool)
    avg_pool = shared_dense_one(avg_pool)
    avg_pool = shared_dense_two(avg_pool)

    max_pool = GlobalMaxPooling2D()(input_feature)
    max_pool = Reshape((1, 1, channel))(max_pool)
    max_pool = shared_dense_one(max_pool)
    max_pool = shared_dense_two(max_pool)

    feature = Add()([avg_pool, max_pool])
    feature = Activation("sigmoid")(feature)

    return Multiply()([input_feature, feature])

def spatial_attention(input_feature, kernel_size=7):
    avg_pool = Lambda(lambda x: tf.reduce_mean(x, axis=-1, keepdims=True))(input_feature)
    max_pool = Lambda(lambda x: tf.reduce_max(x, axis=-1, keepdims=True))(input_feature)
    concat = Concatenate(axis=-1)([avg_pool, max_pool])
    feature = Conv2D(1, kernel_size=kernel_size, padding="same",
                      activation="sigmoid", kernel_initializer="he_normal",
                      use_bias=False)(concat)
    return Multiply()([input_feature, feature])

def cbam_block(input_feature, ratio=8, kernel_size=7):
    x = channel_attention(input_feature, ratio)
    x = spatial_attention(x, kernel_size)
    return x

base_model = InceptionV3(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

x = base_model.output
x = cbam_block(x)
x = GlobalAveragePooling2D()(x)
x = Dense(1024, activation="relu")(x)
x = Dropout(0.5)(x)
x = Dense(512, activation="relu")(x)
x = Dropout(0.5)(x)
output = Dense(NUM_CLASSES, activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    "improved_cbam_inceptionv3_best.keras", monitor="val_accuracy",
    save_best_only=True, mode="max"
)
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_accuracy", patience=10, restore_best_weights=True
)

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=[checkpoint, early_stop]
)

test_loss, test_acc = model.evaluate(test_gen)
print(f"\nIMPROVED (CBAM) Inception-V3 Test Accuracy: {test_acc*100:.2f}%")

model.save("improved_cbam_inceptionv3_final.keras")

Found 1470 images belonging to 21 classes.


Found 315 images belonging to 21 classes.
Found 315 images belonging to 21 classes.



Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_188 (Conv2D) │ (None, 111, 111,  │        864 │ input_layer_2[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 111, 111,  │         96 │ conv2d_188[0][0]  │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_189      │ (None, 111, 111,  │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_189 (Conv2D) │ (None, 109, 109,  │      9,216 │ activation_189[0… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 109, 109,  │         96 │ conv2d_189[0][0]  │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_190      │ (None, 109, 109,  │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_190 (Conv2D) │ (None, 109, 109,  │     18,432 │ activation_190[0… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 109, 109,  │        192 │ conv2d_190[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_191      │ (None, 109, 109,  │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_8     │ (None, 54, 54,    │          0 │ activation_191[0… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_191 (Conv2D) │ (None, 54, 54,    │      5,120 │ max_pooling2d_8[… │
│                     │ 80)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 54, 54,    │        240 │ conv2d_191[0][0]  │
│ (BatchNormalizatio… │ 80)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_192      │ (None, 54, 54,    │          0 │ batch_normalizat… │
│ (Activation)        │ 80)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_192 (Conv2D) │ (None, 52, 52,    │    138,240 │ activation_192[0… │
│                     │ 192)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 52, 52,    │        576 │ conv2d_192[0][0]  │
│ (BatchNormalizatio… │ 192)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_193      │ (None, 52, 52,    │          0 │ batch_normalizat

 Total params: 25,487,511 (97.23 MB)

 Trainable params: 3,684,727 (14.06 MB)

 Non-trainable params: 21,802,784 (83.17 MB)

Epoch 1/50
46/46 ━━━━━━━━━━━━━━━━━━━━ 64s 1s/step - accuracy: 0.0946 - loss: 3.0054 - val_accuracy: 0.4825 - val_loss: 2.6949
Epoch 2/50
46/46 ━━━━━━━━━━━━━━━━━━━━ 51s 1s/step - accuracy: 0.2844 - loss: 2.5913 - val_accuracy: 0.6762 - val_loss: 2.1037
Epoch 3/50
46/46 ━━━━━━━━━━━━━━━━━━━━ 51s 1s/step - accuracy: 0.4204 - loss: 2.0947 - val_accuracy: 0.7524 - val_loss: 1.4165
Epoch 4/50
46/46 ━━━━━━━━━━━━━━━━━━━━ 50s 1s/step - accuracy: 0.5571 - loss: 1.5552 - val_accuracy: 0.8000 - val_loss: 0.9284
Epoch 5/50
46/46 ━━━━━━━━━━━━━━━━━━━━ 50s 1s/step - accuracy: 0.6469 - loss: 1.2006 - val_accuracy: 0.8222 - val_loss: 0.7000
Epoch 6/50
46/46 ━━━━━━━━━━━━━━━━━━━━ 52s 1s/step - accuracy: 0.7054 - loss: 0.9644 - val_accuracy: 0.8540 - val_loss: 0.5444
Epoch 7/50
46/46 ━━━━━━━━━━━━━━━━━━━━ 52s 1s/step - accuracy: 0.7381 - loss: 0.8291 - val_accuracy: 0.8762 - val_loss: 0.4741
Epoch 8/50
46/46 ━━━━━━━━━━━━━━━━━━━━ 52s 1s/step - accuracy: 0.7816 - loss: 0.7172 - val_accuracy: 0.8794 - val_loss:

In [6]:
# step5_improved_cbam_densenet121.py
import tensorflow as tf
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.layers import (
    Dense, Dropout, GlobalAveragePooling2D, GlobalMaxPooling2D,
    Reshape, Multiply, Concatenate, Conv2D, Add, Activation, Lambda
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator

DATA_DIR = "split_dataset"
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 50
NUM_CLASSES = 21

train_datagen = ImageDataGenerator(
    rescale=1./255,
    horizontal_flip=True,
    rotation_range=15,
    zoom_range=0.2
)
val_test_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    f"{DATA_DIR}/train", target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="categorical", shuffle=True
)
val_gen = val_test_datagen.flow_from_directory(
    f"{DATA_DIR}/val", target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="categorical", shuffle=False
)
test_gen = val_test_datagen.flow_from_directory(
    f"{DATA_DIR}/test", target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="categorical", shuffle=False
)

def channel_attention(input_feature, ratio=8):
    channel = input_feature.shape[-1]
    shared_dense_one = Dense(channel // ratio, activation="relu",
                              kernel_initializer="he_normal", use_bias=True)
    shared_dense_two = Dense(channel, kernel_initializer="he_normal", use_bias=True)

    avg_pool = GlobalAveragePooling2D()(input_feature)
    avg_pool = Reshape((1, 1, channel))(avg_pool)
    avg_pool = shared_dense_one(avg_pool)
    avg_pool = shared_dense_two(avg_pool)

    max_pool = GlobalMaxPooling2D()(input_feature)
    max_pool = Reshape((1, 1, channel))(max_pool)
    max_pool = shared_dense_one(max_pool)
    max_pool = shared_dense_two(max_pool)

    feature = Add()([avg_pool, max_pool])
    feature = Activation("sigmoid")(feature)
    return Multiply()([input_feature, feature])

def spatial_attention(input_feature, kernel_size=7):
    avg_pool = Lambda(lambda x: tf.reduce_mean(x, axis=-1, keepdims=True))(input_feature)
    max_pool = Lambda(lambda x: tf.reduce_max(x, axis=-1, keepdims=True))(input_feature)
    concat = Concatenate(axis=-1)([avg_pool, max_pool])
    feature = Conv2D(1, kernel_size=kernel_size, padding="same",
                      activation="sigmoid", kernel_initializer="he_normal",
                      use_bias=False)(concat)
    return Multiply()([input_feature, feature])

def cbam_block(input_feature, ratio=8, kernel_size=7):
    x = channel_attention(input_feature, ratio)
    x = spatial_attention(x, kernel_size)
    return x

base_model = DenseNet121(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

x = base_model.output
x = cbam_block(x)
x = GlobalAveragePooling2D()(x)
x = Dense(1024, activation="relu")(x)
x = Dropout(0.5)(x)
x = Dense(512, activation="relu")(x)
x = Dropout(0.5)(x)
output = Dense(NUM_CLASSES, activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    "improved_cbam_densenet121_best.keras", monitor="val_accuracy",
    save_best_only=True, mode="max"
)
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_accuracy", patience=10, restore_best_weights=True
)

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=[checkpoint, early_stop]
)

test_loss, test_acc = model.evaluate(test_gen)
print(f"\nIMPROVED (CBAM) DenseNet121 Test Accuracy: {test_acc*100:.2f}%")

model.save("improved_cbam_densenet121_final.keras")

Found 1470 images belonging to 21 classes.
Found 315 images belonging to 21 classes.
Found 315 images belonging to 21 classes.
29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 21s 1us/step


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zero_padding2d      │ (None, 230, 230,  │          0 │ input_layer_3[0]… │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,408 │ zero_padding2d[0… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zero_padding2d_1    │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1               │ (None, 56, 56,    │          0 │ zero_padding2d_1… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │        256 │ pool1[0][0]       │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_relu │ (None, 56, 56,    │          0 │ conv2_block1_0_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      8,192 │ conv2_block1_0_r… │
│ (Conv2D)            │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        512 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,864 │ conv2_block1_1_r… │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_concat │ (None, 56, 56,    │          0 │ pool1[0][0],      │
│ (Concatenate)       │ 96)               │            │ conv2_block1_2_c… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block2_0_bn   │ (None, 56, 56,    │        384 │ conv2_block1_con… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block2_0_relu │ (None, 56, 56,    │          0 │ conv2_block2_0_b… │
│ (Activation)        │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block2_1_conv │ (None, 56, 56,    │     12,288 │ conv2_block2_0_r

 Total params: 8,886,071 (33.90 MB)

 Trainable params: 1,848,567 (7.05 MB)

 Non-trainable params: 7,037,504 (26.85 MB)

Epoch 1/50
46/46 ━━━━━━━━━━━━━━━━━━━━ 121s 2s/step - accuracy: 0.0748 - loss: 3.0082 - val_accuracy: 0.3079 - val_loss: 2.8937
Epoch 2/50
46/46 ━━━━━━━━━━━━━━━━━━━━ 105s 2s/step - accuracy: 0.1966 - loss: 2.7971 - val_accuracy: 0.6254 - val_loss: 2.4680
Epoch 3/50
46/46 ━━━━━━━━━━━━━━━━━━━━ 110s 2s/step - accuracy: 0.4231 - loss: 2.2455 - val_accuracy: 0.7651 - val_loss: 1.5450
Epoch 4/50
46/46 ━━━━━━━━━━━━━━━━━━━━ 111s 2s/step - accuracy: 0.5844 - loss: 1.5184 - val_accuracy: 0.7968 - val_loss: 0.8765
Epoch 5/50
46/46 ━━━━━━━━━━━━━━━━━━━━ 114s 2s/step - accuracy: 0.6946 - loss: 1.0570 - val_accuracy: 0.8349 - val_loss: 0.6098
Epoch 6/50
46/46 ━━━━━━━━━━━━━━━━━━━━ 111s 2s/step - accuracy: 0.7224 - loss: 0.8772 - val_accuracy: 0.8413 - val_loss: 0.4906
Epoch 7/50
46/46 ━━━━━━━━━━━━━━━━━━━━ 112s 2s/step - accuracy: 0.7810 - loss: 0.7222 - val_accuracy: 0.8698 - val_loss: 0.4404
Epoch 8/50
46/46 ━━━━━━━━━━━━━━━━━━━━ 114s 2s/step - accuracy: 0.8027 - loss: 0.6457 - val_accuracy: 0.8667 - v

In [9]:
# step6_ensemble_evaluation.py
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import InceptionV3, DenseNet121
from tensorflow.keras.layers import (
    Dense, Dropout, GlobalAveragePooling2D, GlobalMaxPooling2D,
    Reshape, Multiply, Concatenate, Conv2D, Add, Activation, Lambda
)
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

DATA_DIR = "split_dataset"
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 21

# ---------------- CBAM (same as training scripts) ----------------
def channel_attention(input_feature, ratio=8):
    channel = input_feature.shape[-1]
    shared_dense_one = Dense(channel // ratio, activation="relu",
                              kernel_initializer="he_normal", use_bias=True)
    shared_dense_two = Dense(channel, kernel_initializer="he_normal", use_bias=True)

    avg_pool = GlobalAveragePooling2D()(input_feature)
    avg_pool = Reshape((1, 1, channel))(avg_pool)
    avg_pool = shared_dense_one(avg_pool)
    avg_pool = shared_dense_two(avg_pool)

    max_pool = GlobalMaxPooling2D()(input_feature)
    max_pool = Reshape((1, 1, channel))(max_pool)
    max_pool = shared_dense_one(max_pool)
    max_pool = shared_dense_two(max_pool)

    feature = Add()([avg_pool, max_pool])
    feature = Activation("sigmoid")(feature)
    return Multiply()([input_feature, feature])

def spatial_attention(input_feature, kernel_size=7):
    avg_pool = Lambda(lambda x: tf.reduce_mean(x, axis=-1, keepdims=True),
                       output_shape=lambda s: s[:-1] + (1,))(input_feature)
    max_pool = Lambda(lambda x: tf.reduce_max(x, axis=-1, keepdims=True),
                       output_shape=lambda s: s[:-1] + (1,))(input_feature)
    concat = Concatenate(axis=-1)([avg_pool, max_pool])
    feature = Conv2D(1, kernel_size=kernel_size, padding="same",
                      activation="sigmoid", kernel_initializer="he_normal",
                      use_bias=False)(concat)
    return Multiply()([input_feature, feature])

def cbam_block(input_feature, ratio=8, kernel_size=7):
    x = channel_attention(input_feature, ratio)
    x = spatial_attention(x, kernel_size)
    return x

def build_model(backbone_fn):
    base_model = backbone_fn(weights=None, include_top=False, input_shape=(224, 224, 3))
    x = base_model.output
    x = cbam_block(x)
    x = GlobalAveragePooling2D()(x)
    x = Dense(1024, activation="relu")(x)
    x = Dropout(0.5)(x)
    x = Dense(512, activation="relu")(x)
    x = Dropout(0.5)(x)
    output = Dense(NUM_CLASSES, activation="softmax")(x)
    return Model(inputs=base_model.input, outputs=output)

print("Rebuilding architectures and loading weights...")
model_inception = build_model(InceptionV3)
model_inception.load_weights("improved_cbam_inceptionv3_final.keras")

model_densenet = build_model(DenseNet121)
model_densenet.load_weights("improved_cbam_densenet121_final.keras")

# ---------------- Test data ----------------
test_datagen = ImageDataGenerator(rescale=1./255)
test_gen = test_datagen.flow_from_directory(
    f"{DATA_DIR}/test", target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="categorical", shuffle=False
)
class_names = list(test_gen.class_indices.keys())
y_true = test_gen.classes

print("Getting predictions from Inception-V3+CBAM...")
test_gen.reset()
pred_inception = model_inception.predict(test_gen, verbose=1)

print("Getting predictions from DenseNet121+CBAM...")
test_gen.reset()
pred_densenet = model_densenet.predict(test_gen, verbose=1)

acc_inception = accuracy_score(y_true, np.argmax(pred_inception, axis=1))
acc_densenet = accuracy_score(y_true, np.argmax(pred_densenet, axis=1))
print(f"\nInception-V3+CBAM standalone test accuracy: {acc_inception*100:.2f}%")
print(f"DenseNet121+CBAM standalone test accuracy: {acc_densenet*100:.2f}%")

ensemble_pred_avg = (pred_inception + pred_densenet) / 2.0
y_pred_avg = np.argmax(ensemble_pred_avg, axis=1)

w_inception, w_densenet = 0.4, 0.6
ensemble_pred_weighted = (w_inception * pred_inception) + (w_densenet * pred_densenet)
y_pred_weighted = np.argmax(ensemble_pred_weighted, axis=1)

def report(y_true, y_pred, name):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="macro")
    rec = recall_score(y_true, y_pred, average="macro")
    f1 = f1_score(y_true, y_pred, average="macro")
    print(f"\n===== {name} =====")
    print(f"Accuracy:  {acc*100:.2f}%")
    print(f"Precision: {prec*100:.2f}%")
    print(f"Recall:    {rec*100:.2f}%")
    print(f"F1-score:  {f1*100:.2f}%")
    return acc, prec, rec, f1

report(y_true, y_pred_avg, "ENSEMBLE (Simple Average)")
report(y_true, y_pred_weighted, "ENSEMBLE (Weighted Average: 0.4 Inception + 0.6 DenseNet)")

best_pred = y_pred_weighted if accuracy_score(y_true, y_pred_weighted) >= accuracy_score(y_true, y_pred_avg) else y_pred_avg
print("\n===== Detailed Classification Report (Best Ensemble) =====")
print(classification_report(y_true, best_pred, target_names=class_names))

print("\n===== Confusion Matrix (Best Ensemble) =====")
print(confusion_matrix(y_true, best_pred))

Rebuilding architectures and loading weights...
Found 315 images belonging to 21 classes.
Getting predictions from Inception-V3+CBAM...
10/10 ━━━━━━━━━━━━━━━━━━━━ 11s 960ms/step
Getting predictions from DenseNet121+CBAM...
10/10 ━━━━━━━━━━━━━━━━━━━━ 25s 2s/step

Inception-V3+CBAM standalone test accuracy: 92.70%
DenseNet121+CBAM standalone test accuracy: 95.24%

===== ENSEMBLE (Simple Average) =====
Accuracy:  95.56%
Precision: 95.76%
Recall:    95.56%
F1-score:  95.49%

===== ENSEMBLE (Weighted Average: 0.4 Inception + 0.6 DenseNet) =====
Accuracy:  95.87%
Precision: 96.16%
Recall:    95.87%
F1-score:  95.84%

===== Detailed Classification Report (Best Ensemble) =====
                   precision    recall  f1-score   support

     agricultural       1.00      0.93      0.97        15
  baseballdiamond       1.00      1.00      1.00        15
        buildings       0.83      1.00      0.91        15
 denseresidential       0.92      0.73      0.81        15
           harbor       1.

In [11]:
# step7_gradcam.py
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.applications import InceptionV3, DenseNet121
from tensorflow.keras.layers import (
    Dense, Dropout, GlobalAveragePooling2D, GlobalMaxPooling2D,
    Reshape, Multiply, Concatenate, Conv2D, Add, Activation, Lambda
)
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing import image
import os

IMG_SIZE = (224, 224)
NUM_CLASSES = 21
DATA_DIR = "split_dataset/test"

# ---------------- CBAM (same as training) ----------------
def channel_attention(input_feature, ratio=8):
    channel = input_feature.shape[-1]
    shared_dense_one = Dense(channel // ratio, activation="relu",
                              kernel_initializer="he_normal", use_bias=True)
    shared_dense_two = Dense(channel, kernel_initializer="he_normal", use_bias=True)

    avg_pool = GlobalAveragePooling2D()(input_feature)
    avg_pool = Reshape((1, 1, channel))(avg_pool)
    avg_pool = shared_dense_one(avg_pool)
    avg_pool = shared_dense_two(avg_pool)

    max_pool = GlobalMaxPooling2D()(input_feature)
    max_pool = Reshape((1, 1, channel))(max_pool)
    max_pool = shared_dense_one(max_pool)
    max_pool = shared_dense_two(max_pool)

    feature = Add()([avg_pool, max_pool])
    feature = Activation("sigmoid")(feature)
    return Multiply()([input_feature, feature])

def spatial_attention(input_feature, kernel_size=7):
    avg_pool = Lambda(lambda x: tf.reduce_mean(x, axis=-1, keepdims=True),
                       output_shape=lambda s: s[:-1] + (1,))(input_feature)
    max_pool = Lambda(lambda x: tf.reduce_max(x, axis=-1, keepdims=True),
                       output_shape=lambda s: s[:-1] + (1,))(input_feature)
    concat = Concatenate(axis=-1)([avg_pool, max_pool])
    feature = Conv2D(1, kernel_size=kernel_size, padding="same",
                      activation="sigmoid", kernel_initializer="he_normal",
                      use_bias=False)(concat)
    return Multiply()([input_feature, feature])

def cbam_block(input_feature, ratio=8, kernel_size=7):
    x = channel_attention(input_feature, ratio)
    x = spatial_attention(x, kernel_size)
    return x

def build_model(backbone_fn):
    base_model = backbone_fn(weights=None, include_top=False, input_shape=(224, 224, 3))
    x = base_model.output
    cbam_out = cbam_block(x)
    x = GlobalAveragePooling2D()(cbam_out)
    x = Dense(1024, activation="relu")(x)
    x = Dropout(0.5)(x)
    x = Dense(512, activation="relu")(x)
    x = Dropout(0.5)(x)
    output = Dense(NUM_CLASSES, activation="softmax")(x)
    model = Model(inputs=base_model.input, outputs=output)
    return model, cbam_out

print("Rebuilding DenseNet121+CBAM (best individual model) for Grad-CAM...")
model, cbam_output_tensor = build_model(DenseNet121)
model.load_weights("improved_cbam_densenet121_final.keras")

# Grad model: input -> (cbam feature map, final predictions)
grad_model = Model(inputs=model.input, outputs=[cbam_output_tensor, model.output])

def get_gradcam_heatmap(img_array, grad_model, pred_index=None):
    with tf.GradientTape() as tape:
        conv_output, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_output = conv_output[0]
    heatmap = conv_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy(), int(pred_index), predictions[0][pred_index].numpy()

def overlay_heatmap(img_path, heatmap, alpha=0.4):
    img = image.load_img(img_path, target_size=IMG_SIZE)
    img_arr = image.img_to_array(img)

    heatmap_resized = np.uint8(255 * heatmap)
    jet = plt.colormaps["jet"]
    jet_colors = jet(np.arange(256))[:, :3]
    jet_heatmap = jet_colors[heatmap_resized]
    jet_heatmap = image.array_to_img(jet_heatmap)
    jet_heatmap = jet_heatmap.resize(IMG_SIZE)
    jet_heatmap = image.img_to_array(jet_heatmap)

    superimposed = jet_heatmap * alpha + img_arr
    superimposed = image.array_to_img(superimposed)
    return img, superimposed

# ---------------- Pick 1 sample image per class and generate Grad-CAM ----------------
class_names = sorted(os.listdir(DATA_DIR))
os.makedirs("gradcam_outputs", exist_ok=True)

fig, axes = plt.subplots(len(class_names), 2, figsize=(6, 3 * len(class_names)))

for i, cls in enumerate(class_names):
    cls_dir = os.path.join(DATA_DIR, cls)
    sample_file = os.listdir(cls_dir)[0]
    img_path = os.path.join(cls_dir, sample_file)

    img = image.load_img(img_path, target_size=IMG_SIZE)
    img_array = image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    heatmap, pred_idx, pred_conf = get_gradcam_heatmap(img_array, grad_model)
    orig_img, cam_img = overlay_heatmap(img_path, heatmap)

    axes[i, 0].imshow(orig_img)
    axes[i, 0].set_title(f"{cls} (original)", fontsize=8)
    axes[i, 0].axis("off")

    axes[i, 1].imshow(cam_img)
    pred_label = class_names[pred_idx]
    axes[i, 1].set_title(f"Pred: {pred_label} ({pred_conf*100:.0f}%)", fontsize=8)
    axes[i, 1].axis("off")

plt.tight_layout()
plt.savefig("gradcam_outputs/gradcam_all_classes.png", dpi=150, bbox_inches="tight")
print("\nSaved: gradcam_outputs/gradcam_all_classes.png")
plt.close()

print("Grad-CAM generation complete.")

Rebuilding DenseNet121+CBAM (best individual model) for Grad-CAM...

Saved: gradcam_outputs/gradcam_all_classes.png
Grad-CAM generation complete.


In [12]:
# step8_all_figures.py
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.applications import InceptionV3, DenseNet121
from tensorflow.keras.layers import (
    Dense, Dropout, GlobalAveragePooling2D, GlobalMaxPooling2D,
    Reshape, Multiply, Concatenate, Conv2D, Add, Activation, Lambda
)
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
)
import os

DATA_DIR = "split_dataset"
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 21
OUT_DIR = "paper_figures"
os.makedirs(OUT_DIR, exist_ok=True)

# ---------------- CBAM (same as training) ----------------
def channel_attention(input_feature, ratio=8):
    channel = input_feature.shape[-1]
    shared_dense_one = Dense(channel // ratio, activation="relu",
                              kernel_initializer="he_normal", use_bias=True)
    shared_dense_two = Dense(channel, kernel_initializer="he_normal", use_bias=True)

    avg_pool = GlobalAveragePooling2D()(input_feature)
    avg_pool = Reshape((1, 1, channel))(avg_pool)
    avg_pool = shared_dense_one(avg_pool)
    avg_pool = shared_dense_two(avg_pool)

    max_pool = GlobalMaxPooling2D()(input_feature)
    max_pool = Reshape((1, 1, channel))(max_pool)
    max_pool = shared_dense_one(max_pool)
    max_pool = shared_dense_two(max_pool)

    feature = Add()([avg_pool, max_pool])
    feature = Activation("sigmoid")(feature)
    return Multiply()([input_feature, feature])

def spatial_attention(input_feature, kernel_size=7):
    avg_pool = Lambda(lambda x: tf.reduce_mean(x, axis=-1, keepdims=True),
                       output_shape=lambda s: s[:-1] + (1,))(input_feature)
    max_pool = Lambda(lambda x: tf.reduce_max(x, axis=-1, keepdims=True),
                       output_shape=lambda s: s[:-1] + (1,))(input_feature)
    concat = Concatenate(axis=-1)([avg_pool, max_pool])
    feature = Conv2D(1, kernel_size=kernel_size, padding="same",
                      activation="sigmoid", kernel_initializer="he_normal",
                      use_bias=False)(concat)
    return Multiply()([input_feature, feature])

def cbam_block(input_feature, ratio=8, kernel_size=7):
    x = channel_attention(input_feature, ratio)
    x = spatial_attention(x, kernel_size)
    return x

def build_cbam_model(backbone_fn):
    base_model = backbone_fn(weights=None, include_top=False, input_shape=(224, 224, 3))
    x = base_model.output
    x = cbam_block(x)
    x = GlobalAveragePooling2D()(x)
    x = Dense(1024, activation="relu")(x)
    x = Dropout(0.5)(x)
    x = Dense(512, activation="relu")(x)
    x = Dropout(0.5)(x)
    output = Dense(NUM_CLASSES, activation="softmax")(x)
    return Model(inputs=base_model.input, outputs=output)

# ---------------- Test data ----------------
test_datagen = ImageDataGenerator(rescale=1./255)
test_gen = test_datagen.flow_from_directory(
    f"{DATA_DIR}/test", target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="categorical", shuffle=False
)
class_names = list(test_gen.class_indices.keys())
y_true = test_gen.classes

# ---------------- Load all models ----------------
print("Loading baseline Inception-V3...")
model_baseline = load_model("baseline_inceptionv3_final.h5")

print("Loading Inception-V3 + CBAM...")
model_inception_cbam = build_cbam_model(InceptionV3)
model_inception_cbam.load_weights("improved_cbam_inceptionv3_final.keras")

print("Loading DenseNet121 + CBAM...")
model_densenet_cbam = build_cbam_model(DenseNet121)
model_densenet_cbam.load_weights("improved_cbam_densenet121_final.keras")

# ---------------- Get predictions ----------------
def get_preds(model, name):
    test_gen.reset()
    print(f"Predicting: {name}")
    return model.predict(test_gen, verbose=1)

pred_baseline = get_preds(model_baseline, "Baseline Inception-V3")
pred_inception_cbam = get_preds(model_inception_cbam, "Inception-V3 + CBAM")
pred_densenet_cbam = get_preds(model_densenet_cbam, "DenseNet121 + CBAM")

w_inc, w_dense = 0.4, 0.6
pred_ensemble = (w_inc * pred_inception_cbam) + (w_dense * pred_densenet_cbam)

all_models = {
    "Baseline_InceptionV3": pred_baseline,
    "InceptionV3_CBAM": pred_inception_cbam,
    "DenseNet121_CBAM": pred_densenet_cbam,
    "Ensemble_Weighted": pred_ensemble,
}

# ---------------- Metrics table ----------------
metrics_summary = {}
for name, preds in all_models.items():
    y_pred = np.argmax(preds, axis=1)
    acc = accuracy_score(y_true, y_pred) * 100
    prec = precision_score(y_true, y_pred, average="macro") * 100
    rec = recall_score(y_true, y_pred, average="macro") * 100
    f1 = f1_score(y_true, y_pred, average="macro") * 100
    metrics_summary[name] = {"Accuracy": acc, "Precision": prec, "Recall": rec, "F1-Score": f1}
    print(f"{name}: Acc={acc:.2f}% Prec={prec:.2f}% Rec={rec:.2f}% F1={f1:.2f}%")

# ---------------- FIGURE 1: Bar chart comparison ----------------
labels = list(metrics_summary.keys())
metric_types = ["Accuracy", "Precision", "Recall", "F1-Score"]
x = np.arange(len(labels))
width = 0.2

fig, ax = plt.subplots(figsize=(11, 6))
for i, m in enumerate(metric_types):
    values = [metrics_summary[l][m] for l in labels]
    ax.bar(x + i * width, values, width, label=m)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(labels, rotation=15, ha="right")
ax.set_ylabel("Percentage (%)")
ax.set_title("Model Performance Comparison")
ax.set_ylim(0, 100)
ax.legend()
ax.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/fig1_comparison_barchart.png", dpi=200)
plt.close()
print(f"\nSaved: {OUT_DIR}/fig1_comparison_barchart.png")

# ---------------- FIGURE 2: Confusion matrices for ALL models ----------------
def plot_confusion_matrix(y_true, y_pred, class_names, title, save_path):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(12, 10))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(np.arange(len(class_names)))
    ax.set_yticks(np.arange(len(class_names)))
    ax.set_xticklabels(class_names, rotation=90, fontsize=7)
    ax.set_yticklabels(class_names, fontsize=7)
    ax.set_xlabel("Predicted Label")
    ax.set_ylabel("True Label")
    ax.set_title(title)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            val = cm[i, j]
            if val > 0:
                color = "white" if val > cm.max() / 2 else "black"
                ax.text(j, i, str(val), ha="center", va="center", color=color, fontsize=6)

    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.close()

for name, preds in all_models.items():
    y_pred = np.argmax(preds, axis=1)
    save_path = f"{OUT_DIR}/fig2_confusion_matrix_{name}.png"
    plot_confusion_matrix(y_true, y_pred, class_names, f"Confusion Matrix - {name.replace('_', ' ')}", save_path)
    print(f"Saved: {save_path}")

# ---------------- FIGURE 3: Results table as image ----------------
fig, ax = plt.subplots(figsize=(9, 3))
ax.axis("off")
table_data = [["Model"] + metric_types]
for name in labels:
    row = [name.replace("_", " ")] + [f"{metrics_summary[name][m]:.2f}%" for m in metric_types]
    table_data.append(row)

table = ax.table(cellText=table_data, cellLoc="center", loc="center")
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 2)
for j in range(len(metric_types) + 1):
    table[0, j].set_facecolor("#4472C4")
    table[0, j].set_text_props(color="white", weight="bold")

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/fig3_results_table.png", dpi=200, bbox_inches="tight")
plt.close()
print(f"Saved: {OUT_DIR}/fig3_results_table.png")

print("\nAll figures generated successfully in 'paper_figures/' folder.")

Found 315 images belonging to 21 classes.
Loading baseline Inception-V3...


Loading Inception-V3 + CBAM...
Loading DenseNet121 + CBAM...
Predicting: Baseline Inception-V3
10/10 ━━━━━━━━━━━━━━━━━━━━ 11s 906ms/step
Predicting: Inception-V3 + CBAM
10/10 ━━━━━━━━━━━━━━━━━━━━ 10s 902ms/step
Predicting: DenseNet121 + CBAM
10/10 ━━━━━━━━━━━━━━━━━━━━ 24s 2s/step
Baseline_InceptionV3: Acc=92.38% Prec=92.99% Rec=92.38% F1=92.34%
InceptionV3_CBAM: Acc=92.70% Prec=93.23% Rec=92.70% F1=92.70%
DenseNet121_CBAM: Acc=95.24% Prec=95.72% Rec=95.24% F1=95.24%
Ensemble_Weighted: Acc=95.87% Prec=96.16% Rec=95.87% F1=95.84%

Saved: paper_figures/fig1_comparison_barchart.png
Saved: paper_figures/fig2_confusion_matrix_Baseline_InceptionV3.png
Saved: paper_figures/fig2_confusion_matrix_InceptionV3_CBAM.png
Saved: paper_figures/fig2_confusion_matrix_DenseNet121_CBAM.png
Saved: paper_figures/fig2_confusion_matrix_Ensemble_Weighted.png
Saved: paper_figures/fig3_results_table.png

All figures generated successfully in 'paper_figures/' folder.


In [16]:
# step9_auc_roc_stats_distribution.py
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.applications import InceptionV3, DenseNet121
from tensorflow.keras.layers import (
    Dense, Dropout, GlobalAveragePooling2D, GlobalMaxPooling2D,
    Reshape, Multiply, Concatenate, Conv2D, Add, Activation, Lambda
)
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc, roc_auc_score
from statsmodels.stats.contingency_tables import mcnemar
import os

DATA_DIR = "split_dataset"
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 21
OUT_DIR = "paper_figures"
os.makedirs(OUT_DIR, exist_ok=True)

# ---------------- CBAM (same as training) ----------------
def channel_attention(input_feature, ratio=8):
    channel = input_feature.shape[-1]
    shared_dense_one = Dense(channel // ratio, activation="relu",
                              kernel_initializer="he_normal", use_bias=True)
    shared_dense_two = Dense(channel, kernel_initializer="he_normal", use_bias=True)
    avg_pool = GlobalAveragePooling2D()(input_feature)
    avg_pool = Reshape((1, 1, channel))(avg_pool)
    avg_pool = shared_dense_one(avg_pool)
    avg_pool = shared_dense_two(avg_pool)
    max_pool = GlobalMaxPooling2D()(input_feature)
    max_pool = Reshape((1, 1, channel))(max_pool)
    max_pool = shared_dense_one(max_pool)
    max_pool = shared_dense_two(max_pool)
    feature = Add()([avg_pool, max_pool])
    feature = Activation("sigmoid")(feature)
    return Multiply()([input_feature, feature])

def spatial_attention(input_feature, kernel_size=7):
    avg_pool = Lambda(lambda x: tf.reduce_mean(x, axis=-1, keepdims=True),
                       output_shape=lambda s: s[:-1] + (1,))(input_feature)
    max_pool = Lambda(lambda x: tf.reduce_max(x, axis=-1, keepdims=True),
                       output_shape=lambda s: s[:-1] + (1,))(input_feature)
    concat = Concatenate(axis=-1)([avg_pool, max_pool])
    feature = Conv2D(1, kernel_size=kernel_size, padding="same",
                      activation="sigmoid", kernel_initializer="he_normal",
                      use_bias=False)(concat)
    return Multiply()([input_feature, feature])

def cbam_block(input_feature, ratio=8, kernel_size=7):
    x = channel_attention(input_feature, ratio)
    x = spatial_attention(x, kernel_size)
    return x

def build_cbam_model(backbone_fn):
    base_model = backbone_fn(weights=None, include_top=False, input_shape=(224, 224, 3))
    x = base_model.output
    x = cbam_block(x)
    x = GlobalAveragePooling2D()(x)
    x = Dense(1024, activation="relu")(x)
    x = Dropout(0.5)(x)
    x = Dense(512, activation="relu")(x)
    x = Dropout(0.5)(x)
    output = Dense(NUM_CLASSES, activation="softmax")(x)
    return Model(inputs=base_model.input, outputs=output)

# ---------------- Test data ----------------
test_datagen = ImageDataGenerator(rescale=1./255)
test_gen = test_datagen.flow_from_directory(
    f"{DATA_DIR}/test", target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="categorical", shuffle=False
)
class_names = list(test_gen.class_indices.keys())
y_true = test_gen.classes
y_true_bin = label_binarize(y_true, classes=list(range(NUM_CLASSES)))

# ---------------- Load models ----------------
print("Loading models...")
model_baseline = load_model("baseline_inceptionv3_final.h5")

model_inception_cbam = build_cbam_model(InceptionV3)
model_inception_cbam.load_weights("improved_cbam_inceptionv3_final.keras")

model_densenet_cbam = build_cbam_model(DenseNet121)
model_densenet_cbam.load_weights("improved_cbam_densenet121_final.keras")

def get_preds(model, name):
    test_gen.reset()
    print(f"Predicting: {name}")
    return model.predict(test_gen, verbose=1)

pred_baseline = get_preds(model_baseline, "Baseline Inception-V3")
pred_inception_cbam = get_preds(model_inception_cbam, "Inception-V3 + CBAM")
pred_densenet_cbam = get_preds(model_densenet_cbam, "DenseNet121 + CBAM")

w_inc, w_dense = 0.4, 0.6
pred_ensemble = (w_inc * pred_inception_cbam) + (w_dense * pred_densenet_cbam)

all_models = {
    "Baseline_InceptionV3": pred_baseline,
    "InceptionV3_CBAM": pred_inception_cbam,
    "DenseNet121_CBAM": pred_densenet_cbam,
    "Ensemble_Weighted": pred_ensemble,
}

# ==================================================================
# 1. AUC-ROC (macro-average per model + per-class for best model)
# ==================================================================
print("\n===== AUC-ROC SCORES (Macro Average, One-vs-Rest) =====")
auc_summary = {}
for name, preds in all_models.items():
    macro_auc = roc_auc_score(y_true_bin, preds, average="macro", multi_class="ovr")
    micro_auc = roc_auc_score(y_true_bin, preds, average="micro", multi_class="ovr")
    auc_summary[name] = {"macro_auc": macro_auc, "micro_auc": micro_auc}
    print(f"{name}: Macro-AUC={macro_auc:.4f}  Micro-AUC={micro_auc:.4f}")

# Per-class ROC curves for BEST model (Ensemble)
best_preds = pred_ensemble
fig, ax = plt.subplots(figsize=(10, 9))
for i, cls in enumerate(class_names):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], best_preds[:, i])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, lw=1, label=f"{cls} (AUC={roc_auc:.2f})")

ax.plot([0, 1], [0, 1], "k--", lw=1)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("Per-Class ROC Curves - Ensemble Model (One-vs-Rest)")
ax.legend(fontsize=6, loc="lower right", ncol=2)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/fig4_roc_curves_per_class.png", dpi=200)
plt.close()
print(f"\nSaved: {OUT_DIR}/fig4_roc_curves_per_class.png")

# Bar chart of AUC comparison across models
fig, ax = plt.subplots(figsize=(8, 5))
names = list(auc_summary.keys())
macro_aucs = [auc_summary[n]["macro_auc"] for n in names]
ax.bar(names, macro_aucs, color="teal")
ax.set_ylabel("Macro AUC-ROC")
ax.set_title("Macro AUC-ROC Comparison Across Models")
ax.set_ylim(0.9, 1.0)
plt.xticks(rotation=15, ha="right")
for i, v in enumerate(macro_aucs):
    ax.text(i, v + 0.002, f"{v:.4f}", ha="center", fontsize=9)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/fig5_auc_comparison_barchart.png", dpi=200)
plt.close()
print(f"Saved: {OUT_DIR}/fig5_auc_comparison_barchart.png")

# ==================================================================
# 2. STATISTICAL TEST (McNemar's Test - paired comparison, same test set)
# ==================================================================
print("\n===== McNemar's Statistical Significance Test =====")

def mcnemar_test(y_true, pred_a, pred_b, name_a, name_b):
    pred_a_labels = np.argmax(pred_a, axis=1)
    pred_b_labels = np.argmax(pred_b, axis=1)

    a_correct = (pred_a_labels == y_true)
    b_correct = (pred_b_labels == y_true)

    # Contingency table
    both_correct = np.sum(a_correct & b_correct)
    a_only = np.sum(a_correct & ~b_correct)
    b_only = np.sum(~a_correct & b_correct)
    both_wrong = np.sum(~a_correct & ~b_correct)

    table = [[both_correct, a_only], [b_only, both_wrong]]
    result = mcnemar(table, exact=True)

    print(f"\n{name_a} vs {name_b}")
    print(f"  Contingency table: both_correct={both_correct}, {name_a}_only_correct={a_only}, "
          f"{name_b}_only_correct={b_only}, both_wrong={both_wrong}")
    print(f"  McNemar statistic={result.statistic:.4f}, p-value={result.pvalue:.4f}")
    if result.pvalue < 0.05:
        print(f"  -> Statistically SIGNIFICANT difference (p < 0.05)")
    else:
        print(f"  -> NOT statistically significant (p >= 0.05)")
    return result.pvalue

mcnemar_test(y_true, pred_baseline, pred_ensemble, "Baseline_InceptionV3", "Ensemble_Weighted")
mcnemar_test(y_true, pred_baseline, pred_densenet_cbam, "Baseline_InceptionV3", "DenseNet121_CBAM")
mcnemar_test(y_true, pred_inception_cbam, pred_ensemble, "InceptionV3_CBAM", "Ensemble_Weighted")
mcnemar_test(y_true, pred_densenet_cbam, pred_ensemble, "DenseNet121_CBAM", "Ensemble_Weighted")

# ==================================================================
# 3. DATASET / CLASS / TARGET FEATURE DISTRIBUTION
# ==================================================================
print("\n===== Dataset Class Distribution =====")
train_gen_check = ImageDataGenerator(rescale=1./255).flow_from_directory(
    f"{DATA_DIR}/train", target_size=IMG_SIZE, batch_size=1, shuffle=False)
val_gen_check = ImageDataGenerator(rescale=1./255).flow_from_directory(
    f"{DATA_DIR}/val", target_size=IMG_SIZE, batch_size=1, shuffle=False)

train_counts = np.bincount(train_gen_check.classes)
val_counts = np.bincount(val_gen_check.classes)
test_counts = np.bincount(y_true)

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(class_names))
width = 0.25
ax.bar(x - width, train_counts, width, label="Train")
ax.bar(x, val_counts, width, label="Validation")
ax.bar(x + width, test_counts, width, label="Test")
ax.set_xticks(x)
ax.set_xticklabels(class_names, rotation=90, fontsize=7)
ax.set_ylabel("Number of Images")
ax.set_title("Dataset Class Distribution (21 Classes, Target Labels)")
ax.legend()
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/fig6_class_distribution.png", dpi=200)
plt.close()
print(f"Saved: {OUT_DIR}/fig6_class_distribution.png")

print("\n===== Model Input/Output Feature Summary =====")
print(f"Input feature shape (image): (224, 224, 3) -> RGB pixel values normalized [0,1]")
print(f"Target feature (label): categorical, {NUM_CLASSES} classes -> {class_names}")
print(f"Total samples: Train={sum(train_counts)}, Val={sum(val_counts)}, Test={sum(test_counts)}")

print("\nStep 9 complete. All figures saved in 'paper_figures/' folder.")

Found 315 images belonging to 21 classes.
Loading models...


Predicting: Baseline Inception-V3
10/10 ━━━━━━━━━━━━━━━━━━━━ 23s 2s/step 
Predicting: Inception-V3 + CBAM
10/10 ━━━━━━━━━━━━━━━━━━━━ 14s 1s/step 
Predicting: DenseNet121 + CBAM
10/10 ━━━━━━━━━━━━━━━━━━━━ 27s 2s/step

===== AUC-ROC SCORES (Macro Average, One-vs-Rest) =====
Baseline_InceptionV3: Macro-AUC=0.9977  Micro-AUC=0.9980
InceptionV3_CBAM: Macro-AUC=0.9972  Micro-AUC=0.9978
DenseNet121_CBAM: Macro-AUC=0.9989  Micro-AUC=0.9989
Ensemble_Weighted: Macro-AUC=0.9994  Micro-AUC=0.9995

Saved: paper_figures/fig4_roc_curves_per_class.png
Saved: paper_figures/fig5_auc_comparison_barchart.png

===== McNemar's Statistical Significance Test =====

Baseline_InceptionV3 vs Ensemble_Weighted
  Contingency table: both_correct=286, Baseline_InceptionV3_only_correct=5, Ensemble_Weighted_only_correct=16, both_wrong=8
  McNemar statistic=5.0000, p-value=0.0266
  -> Statistically SIGNIFICANT difference (p < 0.05)

Baseline_InceptionV3 vs DenseNet121_CBAM
  Contingency table: both_correct=282, Baselin

In [ ]:
# step10_ablation_study.py
import tensorflow as tf
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.layers import (
    Dense, Dropout, GlobalAveragePooling2D, GlobalMaxPooling2D,
    Reshape, Multiply, Concatenate, Conv2D, Add, Activation, Lambda
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import json

DATA_DIR = "split_dataset"
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 50
NUM_CLASSES = 21

train_datagen = ImageDataGenerator(
    rescale=1./255, horizontal_flip=True, rotation_range=15, zoom_range=0.2
)
val_test_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    f"{DATA_DIR}/train", target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="categorical", shuffle=True
)
val_gen = val_test_datagen.flow_from_directory(
    f"{DATA_DIR}/val", target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="categorical", shuffle=False
)
test_gen = val_test_datagen.flow_from_directory(
    f"{DATA_DIR}/test", target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="categorical", shuffle=False
)

# ---------------- CBAM components ----------------
def channel_attention(input_feature, ratio=8):
    channel = input_feature.shape[-1]
    shared_dense_one = Dense(channel // ratio, activation="relu",
                              kernel_initializer="he_normal", use_bias=True)
    shared_dense_two = Dense(channel, kernel_initializer="he_normal", use_bias=True)
    avg_pool = GlobalAveragePooling2D()(input_feature)
    avg_pool = Reshape((1, 1, channel))(avg_pool)
    avg_pool = shared_dense_one(avg_pool)
    avg_pool = shared_dense_two(avg_pool)
    max_pool = GlobalMaxPooling2D()(input_feature)
    max_pool = Reshape((1, 1, channel))(max_pool)
    max_pool = shared_dense_one(max_pool)
    max_pool = shared_dense_two(max_pool)
    feature = Add()([avg_pool, max_pool])
    feature = Activation("sigmoid")(feature)
    return Multiply()([input_feature, feature])

def spatial_attention(input_feature, kernel_size=7):
    avg_pool = Lambda(lambda x: tf.reduce_mean(x, axis=-1, keepdims=True),
                       output_shape=lambda s: s[:-1] + (1,))(input_feature)
    max_pool = Lambda(lambda x: tf.reduce_max(x, axis=-1, keepdims=True),
                       output_shape=lambda s: s[:-1] + (1,))(input_feature)
    concat = Concatenate(axis=-1)([avg_pool, max_pool])
    feature = Conv2D(1, kernel_size=kernel_size, padding="same",
                      activation="sigmoid", kernel_initializer="he_normal",
                      use_bias=False)(concat)
    return Multiply()([input_feature, feature])

# ---------------- Build model with configurable attention ----------------
def build_model(attention_mode="none"):
    """attention_mode: 'none', 'channel_only', 'spatial_only', 'full_cbam'"""
    base_model = DenseNet121(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
    base_model.trainable = False

    x = base_model.output
    if attention_mode == "channel_only":
        x = channel_attention(x)
    elif attention_mode == "spatial_only":
        x = spatial_attention(x)
    elif attention_mode == "full_cbam":
        x = channel_attention(x)
        x = spatial_attention(x)
    # "none" -> no attention, pass through

    x = GlobalAveragePooling2D()(x)
    x = Dense(1024, activation="relu")(x)
    x = Dropout(0.5)(x)
    x = Dense(512, activation="relu")(x)
    x = Dropout(0.5)(x)
    output = Dense(NUM_CLASSES, activation="softmax")(x)
    return Model(inputs=base_model.input, outputs=output)

# ---------------- Train each ablation variant ----------------
def train_variant(attention_mode, save_name):
    print(f"\n{'='*60}\nTraining variant: {attention_mode}\n{'='*60}")
    model = build_model(attention_mode)
    model.compile(optimizer=Adam(learning_rate=0.0001),
                  loss="categorical_crossentropy", metrics=["accuracy"])

    checkpoint = tf.keras.callbacks.ModelCheckpoint(
        f"{save_name}_best.keras", monitor="val_accuracy", save_best_only=True, mode="max")
    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=10, restore_best_weights=True)

    model.fit(train_gen, validation_data=val_gen, epochs=EPOCHS,
              callbacks=[checkpoint, early_stop], verbose=1)

    test_loss, test_acc = model.evaluate(test_gen)
    print(f"\n{attention_mode} Test Accuracy: {test_acc*100:.2f}%")
    model.save_weights(f"{save_name}_final.keras")
    return test_acc

results = {}

# 1. No attention (plain DenseNet121)
results["DenseNet121_NoAttention"] = train_variant("none", "ablation_densenet_none")

# 2. Channel attention only
results["DenseNet121_ChannelOnly"] = train_variant("channel_only", "ablation_densenet_channel")

# 3. Spatial attention only
results["DenseNet121_SpatialOnly"] = train_variant("spatial_only", "ablation_densenet_spatial")

# 4. Full CBAM result already known: 95.24% (from earlier training)
results["DenseNet121_FullCBAM"] = 0.9524

print("\n\n===== ABLATION STUDY SUMMARY =====")
for name, acc in results.items():
    print(f"{name}: {acc*100:.2f}%")

with open("ablation_results.json", "w") as f:
    json.dump(results, f, indent=2)
print("\nSaved: ablation_results.json")

In [3]:
# step10_ablation_study_remaining.py
import tensorflow as tf
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.layers import (
    Dense, Dropout, GlobalAveragePooling2D, GlobalMaxPooling2D,
    Reshape, Multiply, Concatenate, Conv2D, Add, Activation, Lambda
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import json

DATA_DIR = "split_dataset"
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 50
NUM_CLASSES = 21

train_datagen = ImageDataGenerator(
    rescale=1./255, horizontal_flip=True, rotation_range=15, zoom_range=0.2
)
val_test_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    f"{DATA_DIR}/train", target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="categorical", shuffle=True
)
val_gen = val_test_datagen.flow_from_directory(
    f"{DATA_DIR}/val", target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="categorical", shuffle=False
)
test_gen = val_test_datagen.flow_from_directory(
    f"{DATA_DIR}/test", target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="categorical", shuffle=False
)

def channel_attention(input_feature, ratio=8):
    channel = input_feature.shape[-1]
    shared_dense_one = Dense(channel // ratio, activation="relu",
                              kernel_initializer="he_normal", use_bias=True)
    shared_dense_two = Dense(channel, kernel_initializer="he_normal", use_bias=True)
    avg_pool = GlobalAveragePooling2D()(input_feature)
    avg_pool = Reshape((1, 1, channel))(avg_pool)
    avg_pool = shared_dense_one(avg_pool)
    avg_pool = shared_dense_two(avg_pool)
    max_pool = GlobalMaxPooling2D()(input_feature)
    max_pool = Reshape((1, 1, channel))(max_pool)
    max_pool = shared_dense_one(max_pool)
    max_pool = shared_dense_two(max_pool)
    feature = Add()([avg_pool, max_pool])
    feature = Activation("sigmoid")(feature)
    return Multiply()([input_feature, feature])

def spatial_attention(input_feature, kernel_size=7):
    avg_pool = Lambda(lambda x: tf.reduce_mean(x, axis=-1, keepdims=True),
                       output_shape=lambda s: s[:-1] + (1,))(input_feature)
    max_pool = Lambda(lambda x: tf.reduce_max(x, axis=-1, keepdims=True),
                       output_shape=lambda s: s[:-1] + (1,))(input_feature)
    concat = Concatenate(axis=-1)([avg_pool, max_pool])
    feature = Conv2D(1, kernel_size=kernel_size, padding="same",
                      activation="sigmoid", kernel_initializer="he_normal",
                      use_bias=False)(concat)
    return Multiply()([input_feature, feature])

def build_model(attention_mode="none"):
    base_model = DenseNet121(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
    base_model.trainable = False
    x = base_model.output
    if attention_mode == "channel_only":
        x = channel_attention(x)
    elif attention_mode == "spatial_only":
        x = spatial_attention(x)
    elif attention_mode == "full_cbam":
        x = channel_attention(x)
        x = spatial_attention(x)
    x = GlobalAveragePooling2D()(x)
    x = Dense(1024, activation="relu")(x)
    x = Dropout(0.5)(x)
    x = Dense(512, activation="relu")(x)
    x = Dropout(0.5)(x)
    output = Dense(NUM_CLASSES, activation="softmax")(x)
    return Model(inputs=base_model.input, outputs=output)

def train_variant(attention_mode, save_name):
    print(f"\n{'='*60}\nTraining variant: {attention_mode}\n{'='*60}")
    model = build_model(attention_mode)
    model.compile(optimizer=Adam(learning_rate=0.0001),
                  loss="categorical_crossentropy", metrics=["accuracy"])

    checkpoint = tf.keras.callbacks.ModelCheckpoint(
        f"{save_name}_best.keras", monitor="val_accuracy", save_best_only=True, mode="max")
    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=10, restore_best_weights=True)

    model.fit(train_gen, validation_data=val_gen, epochs=EPOCHS,
              callbacks=[checkpoint, early_stop], verbose=1)

    test_loss, test_acc = model.evaluate(test_gen)
    print(f"\n{attention_mode} Test Accuracy: {test_acc*100:.2f}%")
    model.save_weights(f"{save_name}_final.weights.h5")   # <-- fixed extension
    return test_acc

results = {}
results["DenseNet121_NoAttention"] = 0.9460   # already trained, known result

results["DenseNet121_ChannelOnly"] = train_variant("channel_only", "ablation_densenet_channel")
results["DenseNet121_SpatialOnly"] = train_variant("spatial_only", "ablation_densenet_spatial")
results["DenseNet121_FullCBAM"] = 0.9524   # already known from earlier training

print("\n\n===== ABLATION STUDY SUMMARY =====")
for name, acc in results.items():
    print(f"{name}: {acc*100:.2f}%")

with open("ablation_results.json", "w") as f:
    json.dump(results, f, indent=2)
print("\nSaved: ablation_results.json")

Found 1470 images belonging to 21 classes.
Found 315 images belonging to 21 classes.
Found 315 images belonging to 21 classes.

Training variant: channel_only
Epoch 1/50
46/46 ━━━━━━━━━━━━━━━━━━━━ 129s 2s/step - accuracy: 0.0837 - loss: 3.0449 - val_accuracy: 0.4984 - val_loss: 2.6252
Epoch 2/50
46/46 ━━━━━━━━━━━━━━━━━━━━ 108s 2s/step - accuracy: 0.2667 - loss: 2.5712 - val_accuracy: 0.6317 - val_loss: 2.1419
Epoch 3/50
46/46 ━━━━━━━━━━━━━━━━━━━━ 108s 2s/step - accuracy: 0.4259 - loss: 2.1303 - val_accuracy: 0.7365 - val_loss: 1.5604
Epoch 4/50
46/46 ━━━━━━━━━━━━━━━━━━━━ 108s 2s/step - accuracy: 0.5429 - loss: 1.6715 - val_accuracy: 0.7810 - val_loss: 1.1029
Epoch 5/50
46/46 ━━━━━━━━━━━━━━━━━━━━ 110s 2s/step - accuracy: 0.6245 - loss: 1.3464 - val_accuracy: 0.8222 - val_loss: 0.8153
Epoch 6/50
46/46 ━━━━━━━━━━━━━━━━━━━━ 110s 2s/step - accuracy: 0.6844 - loss: 1.0919 - val_accuracy: 0.8540 - val_loss: 0.6450
Epoch 7/50
46/46 ━━━━━━━━━━━━━━━━━━━━ 112s 2s/step - accuracy: 0.7102 - loss: 0

In [4]:
# step10b_ablation_chart.py
import json
import matplotlib.pyplot as plt

with open("ablation_results.json") as f:
    results = json.load(f)

names = list(results.keys())
accs = [results[n] * 100 for n in names]
labels = ["No Attention", "Channel Only", "Spatial Only", "Full CBAM"]
colors = ["gray", "green", "orange", "steelblue"]

fig, ax = plt.subplots(figsize=(8, 6))
bars = ax.bar(labels, accs, color=colors)
ax.set_ylabel("Test Accuracy (%)")
ax.set_title("Ablation Study: Contribution of CBAM Components (DenseNet121 Backbone)")
ax.set_ylim(90, 97)
ax.grid(axis="y", linestyle="--", alpha=0.5)

for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, acc + 0.1, f"{acc:.2f}%",
            ha="center", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.savefig("paper_figures/fig7_ablation_study.png", dpi=200)
plt.close()
print("Saved: paper_figures/fig7_ablation_study.png")

Saved: paper_figures/fig7_ablation_study.png


In [1]:
# step11_shap_analysis.py
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import shap
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.layers import (
    Dense, Dropout, GlobalAveragePooling2D, GlobalMaxPooling2D,
    Reshape, Multiply, Concatenate, Conv2D, Add, Activation, Lambda
)
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.preprocessing import image
import os

IMG_SIZE = (224, 224)
NUM_CLASSES = 21
DATA_DIR = "split_dataset"
OUT_DIR = "paper_figures"
os.makedirs(OUT_DIR, exist_ok=True)

# ---------------- CBAM (same as training) ----------------
def channel_attention(input_feature, ratio=8):
    channel = input_feature.shape[-1]
    shared_dense_one = Dense(channel // ratio, activation="relu",
                              kernel_initializer="he_normal", use_bias=True)
    shared_dense_two = Dense(channel, kernel_initializer="he_normal", use_bias=True)
    avg_pool = GlobalAveragePooling2D()(input_feature)
    avg_pool = Reshape((1, 1, channel))(avg_pool)
    avg_pool = shared_dense_one(avg_pool)
    avg_pool = shared_dense_two(avg_pool)
    max_pool = GlobalMaxPooling2D()(input_feature)
    max_pool = Reshape((1, 1, channel))(max_pool)
    max_pool = shared_dense_one(max_pool)
    max_pool = shared_dense_two(max_pool)
    feature = Add()([avg_pool, max_pool])
    feature = Activation("sigmoid")(feature)
    return Multiply()([input_feature, feature])

def spatial_attention(input_feature, kernel_size=7):
    avg_pool = Lambda(lambda x: tf.reduce_mean(x, axis=-1, keepdims=True),
                       output_shape=lambda s: s[:-1] + (1,))(input_feature)
    max_pool = Lambda(lambda x: tf.reduce_max(x, axis=-1, keepdims=True),
                       output_shape=lambda s: s[:-1] + (1,))(input_feature)
    concat = Concatenate(axis=-1)([avg_pool, max_pool])
    feature = Conv2D(1, kernel_size=kernel_size, padding="same",
                      activation="sigmoid", kernel_initializer="he_normal",
                      use_bias=False)(concat)
    return Multiply()([input_feature, feature])

def cbam_block(input_feature, ratio=8, kernel_size=7):
    x = channel_attention(input_feature, ratio)
    x = spatial_attention(x, kernel_size)
    return x

def build_cbam_model():
    base_model = DenseNet121(weights=None, include_top=False, input_shape=(224, 224, 3))
    x = base_model.output
    x = cbam_block(x)
    x = GlobalAveragePooling2D()(x)
    x = Dense(1024, activation="relu")(x)
    x = Dropout(0.5)(x)
    x = Dense(512, activation="relu")(x)
    x = Dropout(0.5)(x)
    output = Dense(NUM_CLASSES, activation="softmax")(x)
    return Model(inputs=base_model.input, outputs=output)

print("Loading DenseNet121+CBAM model...")
model = build_cbam_model()
model.load_weights("improved_cbam_densenet121_final.keras")

# ---------------- Background samples (for SHAP baseline) ----------------
test_datagen = ImageDataGenerator(rescale=1./255)
background_gen = test_datagen.flow_from_directory(
    f"{DATA_DIR}/test", target_size=IMG_SIZE, batch_size=32,
    class_mode="categorical", shuffle=True, seed=42
)
background_batch, _ = next(background_gen)
background = background_batch[:20]   # 20 images as background reference

print("Building SHAP GradientExplainer (this may take a few minutes)...")
explainer = shap.GradientExplainer(model, background)

# ---------------- Pick a few sample images (1 per selected class) to explain ----------------
class_names = sorted(os.listdir(f"{DATA_DIR}/test"))
sample_classes = class_names[:6]   # limit to 6 classes to keep it fast; increase if needed

images_to_explain = []
labels_for_images = []

for cls in sample_classes:
    cls_dir = os.path.join(f"{DATA_DIR}/test", cls)
    sample_file = os.listdir(cls_dir)[0]
    img_path = os.path.join(cls_dir, sample_file)
    img = image.load_img(img_path, target_size=IMG_SIZE)
    img_array = image.img_to_array(img) / 255.0
    images_to_explain.append(img_array)
    labels_for_images.append(cls)

images_to_explain = np.array(images_to_explain)

print(f"Computing SHAP values for {len(images_to_explain)} sample images...")
shap_values = explainer.shap_values(images_to_explain)

# ---------------- Plot SHAP results ----------------
predictions = model.predict(images_to_explain)
predicted_classes = np.argmax(predictions, axis=1)

fig, axes = plt.subplots(len(sample_classes), 2, figsize=(8, 4 * len(sample_classes)))

for i in range(len(sample_classes)):
    pred_idx = predicted_classes[i]
    # shap_values shape: list of arrays (one per class) OR array with extra class dim
    if isinstance(shap_values, list):
        sv = shap_values[pred_idx][i]
    else:
        sv = shap_values[i, ..., pred_idx]

    sv_sum = np.sum(np.abs(sv), axis=-1)   # aggregate across RGB channels

    axes[i, 0].imshow(images_to_explain[i])
    axes[i, 0].set_title(f"True: {labels_for_images[i]}", fontsize=9)
    axes[i, 0].axis("off")

    axes[i, 1].imshow(images_to_explain[i])
    im = axes[i, 1].imshow(sv_sum, cmap="jet", alpha=0.6)
    axes[i, 1].set_title(f"SHAP - Pred: {class_names[pred_idx]}", fontsize=9)
    axes[i, 1].axis("off")

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/fig8_shap_analysis.png", dpi=150, bbox_inches="tight")
plt.close()
print(f"\nSaved: {OUT_DIR}/fig8_shap_analysis.png")
print("SHAP analysis complete.")

Loading DenseNet121+CBAM model...
Found 315 images belonging to 21 classes.
Building SHAP GradientExplainer (this may take a few minutes)...
Computing SHAP values for 6 sample images...


c:\Users\Maaz Muhammad\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\models\functional.py:259: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: keras_tensor
Received: inputs=['Tensor(shape=(6, 224, 224, 3))']
  warnings.warn(msg)
c:\Users\Maaz Muhammad\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\models\functional.py:259: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: keras_tensor
Received: inputs=['Tensor(shape=(50, 224, 224, 3))']
  warnings.warn(msg)


1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step

Saved: paper_figures/fig8_shap_analysis.png
SHAP analysis complete.
